# 01 — Read and safely join the raw tables

This notebook reads every Assignment 01 table at its native grain. It profiles keys, duplicates, and missingness before joining. All one-to-many sources are reduced to one row per order first, preventing silent row multiplication.

In [1]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert ROOT.name == "02-olist-late-delivery-ml"
SEED = 42
pd.set_option("display.max_columns", 100)
from sqlalchemy import create_engine, URL

url = URL.create(
    "postgresql+psycopg",
    username=os.getenv("POSTGRES_USER", "olist"),
    password=os.getenv("POSTGRES_PASSWORD", "olist_dev_password"),
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    database=os.getenv("POSTGRES_DB", "olist"),
)
engine = create_engine(url)
TABLES = [
    "category_translation", "customers", "geolocation", "order_items",
    "order_payments", "order_reviews", "orders", "products", "sellers",
]
with engine.connect() as connection:
    frames = {
        table: pd.read_sql_table(table, connection, schema="raw")
        for table in TABLES
    }
assert len(frames["orders"]) == 99_441
pd.DataFrame({name: frame.shape for name, frame in frames.items()}, index=["rows", "columns"]).T

,rows,columns
category_translation,71,2
customers,99441,5
geolocation,1000163,5
order_items,112650,7
order_payments,103886,5
order_reviews,100000,7
orders,99441,8
products,32951,9
sellers,3095,4


## Native grains and quality checks

`geolocation` is intentionally not unique by ZIP prefix, and review IDs are not guaranteed unique. Composite keys are checked where appropriate. This profile is join-focused rather than a substitute for the later training-only EDA.

In [2]:
keys = {
    "category_translation": ["product_category_name"],
    "customers": ["customer_id"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "orders": ["order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
}
grains = {
    "category_translation": "one category translation",
    "customers": "one order-specific customer record",
    "geolocation": "one observed coordinate/city for a ZIP prefix",
    "order_items": "one item position in an order",
    "order_payments": "one payment transaction in an order",
    "order_reviews": "one submitted review record",
    "orders": "one order",
    "products": "one product",
    "sellers": "one seller",
}
profile = {}
for name, frame in frames.items():
    likely_key = keys[name]
    profile[name] = {
        "rows": len(frame),
        "columns": len(frame.columns),
        "likely_keys": likely_key,
        "key_unique": not frame.duplicated(likely_key).any(),
        "exact_duplicate_rows": int(frame.duplicated().sum()),
        "grain": grains[name],
        "missing": {c: int(n) for c, n in frame.isna().sum().items() if n},
    }
profile

{'category_translation': {'rows': 71,
  'columns': 2,
  'likely_keys': ['product_category_name'],
  'key_unique': True,
  'exact_duplicate_rows': 0,
  'grain': 'one category translation',
  'missing': {}},
 'customers': {'rows': 99441,
  'columns': 5,
  'likely_keys': ['customer_id'],
  'key_unique': True,
  'exact_duplicate_rows': 0,
  'grain': 'one order-specific customer record',
  'missing': {}},
 'geolocation': {'rows': 1000163,
  'columns': 5,
  'likely_keys': ['geolocation_zip_code_prefix'],
  'key_unique': False,
  'exact_duplicate_rows': 261831,
  'grain': 'one observed coordinate/city for a ZIP prefix',
  'missing': {}},
 'order_items': {'rows': 112650,
  'columns': 7,
  'likely_keys': ['order_id', 'order_item_id'],
  'key_unique': True,
  'exact_duplicate_rows': 0,
  'grain': 'one item position in an order',
  'missing': {}},
 'order_payments': {'rows': 103886,
  'columns': 5,
  'likely_keys': ['order_id', 'payment_sequential'],
  'key_unique': True,
  'exact_duplicate_rows'

## Deterministic geography preparation

ZIP prefixes have many coordinate observations, so median latitude/longitude provides one robust representative point per prefix. Raw medians are retained for later train-only quality analysis. A broad published-domain check (`latitude -34..6`, `longitude -74..-34`) identifies points outside Brazil; invalid endpoints become missing before distance calculation.

Distance is computed per item/seller with Haversine, then aggregated to the order mean and maximum. This correctly handles multi-seller orders and uses no outcome information.

In [3]:
geolocation = frames["geolocation"]
geo = (
    geolocation.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geo_lat=("geolocation_lat", "median"),
        geo_lng=("geolocation_lng", "median"),
    )
)

customers = frames["customers"].merge(
    geo,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
).drop(columns="geolocation_zip_code_prefix")

sellers = frames["sellers"].merge(
    geo,
    left_on="seller_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
).drop(columns="geolocation_zip_code_prefix")
sellers = sellers.rename(columns={"geo_lat": "seller_lat", "geo_lng": "seller_lng"})

products = frames["products"].merge(
    frames["category_translation"],
    on="product_category_name",
    how="left",
    validate="many_to_one",
)

items = frames["order_items"].merge(
    products, on="product_id", how="left", validate="many_to_one"
)
items = items.merge(
    sellers, on="seller_id", how="left", validate="many_to_one"
)
items = items.merge(
    frames["orders"][["order_id", "customer_id"]],
    on="order_id",
    how="left",
    validate="many_to_one",
)
items = items.merge(
    customers[["customer_id", "customer_state", "geo_lat", "geo_lng"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)

customer_valid = items.geo_lat.between(-34, 6) & items.geo_lng.between(-74, -34)
seller_valid = items.seller_lat.between(-34, 6) & items.seller_lng.between(-74, -34)
valid_pair = customer_valid & seller_valid

lat1 = np.radians(items.geo_lat)
lat2 = np.radians(items.seller_lat)
dlat = lat2 - lat1
dlng = np.radians(items.seller_lng - items.geo_lng)
a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2) ** 2
items["customer_seller_distance_km"] = np.where(
    valid_pair, 6371.0088 * 2 * np.arcsin(np.sqrt(a)), np.nan
)
items["same_state"] = (items.customer_state == items.seller_state).astype(float)
items.loc[items.customer_state.isna() | items.seller_state.isna(), "same_state"] = np.nan
items["product_volume_cm3"] = (
    items.product_length_cm * items.product_height_cm * items.product_width_cm
)
{
    "representative_zip_points": len(geo),
    "invalid_customer_item_endpoints": int((~customer_valid & items.geo_lat.notna()).sum()),
    "invalid_seller_item_endpoints": int((~seller_valid & items.seller_lat.notna()).sum()),
    "valid_distance_items": int(items.customer_seller_distance_km.notna().sum()),
}

{'representative_zip_points': 19015,
 'invalid_customer_item_endpoints': 4,
 'invalid_seller_item_endpoints': 0,
 'valid_distance_items': 112092}

## Order-level item and payment aggregation

Modes summarize dominant category/state/type deterministically; numeric aggregates retain order size and value. `shipping_limit_date` is not propagated because its exact availability at purchase is not established. Reviews are profiled above but never joined because they are post-delivery.

In [4]:
def mode_or_missing(series):
    modes = series.dropna().mode()
    return modes.iloc[0] if len(modes) else np.nan

item_agg = (
    items.groupby("order_id", as_index=False)
    .agg(
        item_count=("order_item_id", "count"),
        unique_product_count=("product_id", "nunique"),
        unique_seller_count=("seller_id", "nunique"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        mean_product_weight_g=("product_weight_g", "mean"),
        mean_product_volume_cm3=("product_volume_cm3", "mean"),
        mean_product_photos_qty=("product_photos_qty", "mean"),
        dominant_product_category=("product_category_name_english", mode_or_missing),
        dominant_seller_state=("seller_state", mode_or_missing),
        dominant_seller_zip_prefix=("seller_zip_code_prefix", mode_or_missing),
        same_state_share=("same_state", "mean"),
        mean_customer_seller_distance_km=("customer_seller_distance_km", "mean"),
        max_customer_seller_distance_km=("customer_seller_distance_km", "max"),
    )
)

payment_agg = (
    frames["order_payments"].groupby("order_id", as_index=False)
    .agg(
        payment_count=("payment_sequential", "count"),
        payment_value=("payment_value", "sum"),
        max_payment_installments=("payment_installments", "max"),
        dominant_payment_type=("payment_type", mode_or_missing),
    )
)

joined = frames["orders"].merge(
    customers, on="customer_id", how="left", validate="many_to_one"
)
joined = joined.merge(item_agg, on="order_id", how="left", validate="one_to_one")
joined = joined.merge(payment_agg, on="order_id", how="left", validate="one_to_one")
joined.shape

(99441, 32)

## Grain, roles, and saved contract

The assertions make join explosion impossible to overlook. Future/outcome fields remain only for labeling and descriptive EDA; Notebook 05 controls model admission with a whitelist.

In [5]:
roles = {
    "identifiers": ["order_id", "customer_id", "customer_unique_id"],
    "future_or_label_only": [
        "order_status", "order_approved_at", "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ],
    "candidate_features": [
        c for c in joined.columns
        if c not in {
            "order_id", "customer_id", "customer_unique_id", "order_status",
            "order_approved_at", "order_delivered_carrier_date",
            "order_delivered_customer_date",
        }
    ],
}
assert joined.order_id.is_unique
assert len(joined) == joined.order_id.nunique() == len(frames["orders"]) == 99_441
assert not any(c.startswith("review_") for c in joined)

out = ROOT / "artifacts/01_joined"
out.mkdir(parents=True, exist_ok=True)
joined.to_parquet(out / "ml_orders.parquet", index=False)
(out / "validation.json").write_text(json.dumps({
    "table_profiles": profile,
    "final_rows": len(joined),
    "unique_orders": joined.order_id.nunique(),
    "column_roles": roles,
}, indent=2, default=str))
{"shape": joined.shape, "artifact": str(out / "ml_orders.parquet")}

{'shape': (99441, 32),
 'artifact': '/home/waleed/Qafza/Qafza-AI-ML-Camp/assignments/02-olist-late-delivery-ml/artifacts/01_joined/ml_orders.parquet'}